# Pure Theoretical CMB Maps — Parameter Variants

Extends `generate_cmb_maps_theory_baseline` by sampling cosmological parameters from three physically motivated posteriors.

**Same theoretical idealisations throughout:**
- Unlensed scalar power spectra from CAMB
- No beam, no instrumental noise, no galactic mask
- BB = 0 (no primordial tensors, no lensing-induced B-modes)
- TB = EB = 0 (parity symmetry)
- Only stochasticity: cosmic variance from a single Gaussian random realisation per seed

**Three simulation sets**

| Set | Folder | Cosmology | Priors |
|---|---|---|---|
| A | `theory_planck_posterior` | ΛCDM | Planck 2018 TT,TE,EE+lowE+lensing ±3σ |
| B | `theory_desi_w0wa` | w0waCDM | DESI 2024 BAO+CMB+DES-SN5YR ±3σ, w₀/wₐ jointly sampled |
| C | `theory_feature` | ΛCDM + oscillatory feature | Planck best-fit ΛCDM + feature priors |

Each set produces 1000 realisations: 5 FITS maps (T, Q, U, E, B) + metadata CSV.

**Seed scheme:** realisation `i` uses `seed + i` for the map draw across all sets, so same-index maps across sets differ only through the power spectrum, not the noise realisation.

In [ ]:
import os
import csv
import json
import datetime
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import camb

from collections import namedtuple
from tqdm.notebook import tqdm
from getdist import loadMCSamples
from skysimulation import PK as _PK

print(f'CAMB version: {camb.__version__}')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. Configuration

In [ ]:
# ── Resolution ────────────────────────────────────────────────────────────────
nside  = 256
lmax   = 3 * nside   # natural limit for nside=256
seed   = 314100
N_real = 1000

# ── Planck 2018 best-fit ΛCDM (Table 2, arXiv:1807.06209) — reference values ─
planck_bf = dict(
    H0    = 67.36,
    ombh2 = 0.02237,
    omch2 = 0.1200,
    mnu   = 0.06,
    omk   = 0.0,
    tau   = 0.0544,
    As    = 2.101e-9,
    ns    = 0.9649,
)

# ── Output directories ────────────────────────────────────────────────────────
_maps_base = '/nvme/h/lchristodoulou/data_p318'

out_planck_post = os.path.join(_maps_base, 'theory_planck_posterior')
out_desi_w0wa   = os.path.join(_maps_base, 'theory_desi_w0wa')
out_feature     = os.path.join(_maps_base, 'theory_feature')

for d in [out_planck_post, out_desi_w0wa, out_feature]:
    os.makedirs(d, exist_ok=True)

print(f'nside={nside}  lmax={lmax}  seed={seed}  N_real={N_real}')
print(f'Outputs under: {_maps_base}/')

## 2. Helper Functions

### 2.1 Unlensed spectra with optional dark energy and feature PK

`generate_unlensed_spectra` wraps CAMB's unlensed scalar output and adds support for:
- CPL dark energy $w(a) = w_0 + w_a(1-a)$ via `DarkEnergyPPF` (handles phantom crossing $w < -1$)
- A custom primordial power spectrum feature via the `PK` class

Returns $D_\ell = \frac{\ell(\ell+1)}{2\pi}C_\ell$ in $\mu K^2$ for TT, TE, EE as a named tuple, indexed from $\ell=0$.

### 2.2 $D_\ell \to C_\ell$ conversion

$$C_\ell = \frac{2\pi}{\ell(\ell+1)}D_\ell, \quad \ell \ge 2; \qquad C_0 = C_1 = 0$$

### 2.3 Map synthesis and saving

`make_and_save_theory_maps` draws T, Q, U from a Gaussian random field, derives E and B via
spin-2 harmonic decomposition, and writes five float32 FITS files. No beam, no mask applied.

In [ ]:
_CMBUnlensed = namedtuple('CMBUnlensed', ['ell', 'tt', 'te', 'ee'])

def generate_unlensed_spectra(
    H0, ombh2, omch2, mnu, omk, tau, As, ns, lmax,
    w0=-1.0, wa=0.0,
    custom_PK=False, amp=0, freq=0, wid=1, centre=0.05, phase=0,
):
    """Unlensed scalar CMB D_ell spectra (μK²) with CPL dark energy and optional feature PK."""
    params = camb.set_params(
        H0=H0, ombh2=ombh2, omch2=omch2, mnu=mnu, omk=omk, tau=tau,
        As=As, ns=ns, lmax=lmax,
    )
    params.DarkEnergy = camb.dark_energy.DarkEnergyPPF()
    params.DarkEnergy.set_params(w=w0, wa=wa)
    if custom_PK:
        params.set_initial_power_function(
            _PK, args=(As, ns, amp, freq, wid, centre, phase),
            effective_ns_for_nonlinear=ns,
        )
    else:
        params.InitPower.set_params(As=As, ns=ns)
    results  = camb.get_results(params)
    unlensed = results.get_cmb_power_spectra(params, CMB_unit='muK')['unlensed_scalar']
    ell = np.arange(len(unlensed))
    return _CMBUnlensed(ell, unlensed[:, 0], unlensed[:, 3], unlensed[:, 1])


def dl_to_cl(ell_arr, dl_arr):
    """D_ell -> C_ell; ell=0,1 set to zero."""
    cl = np.zeros_like(dl_arr)
    mask = ell_arr >= 2
    cl[mask] = 2 * np.pi / (ell_arr[mask] * (ell_arr[mask] + 1)) * dl_arr[mask]
    return cl


def make_and_save_theory_maps(cl_tt, cl_ee, cl_te, nside, seed_i, out_dir, tag):
    """Synthesise pure-theory maps (no beam, no noise, no mask); save T,Q,U,E,B as FITS."""
    n = len(cl_tt)
    np.random.seed(seed_i)
    T, Q, U = hp.synfast(
        [cl_tt, cl_ee, np.zeros(n), cl_te, np.zeros(n), np.zeros(n)],
        nside=nside, new=True, pol=True,
    )
    _, alm_E, alm_B = hp.map2alm([T, Q, U], pol=True)
    E = hp.alm2map(alm_E, nside=nside)
    B = hp.alm2map(alm_B, nside=nside)
    for comp, m in [('T', T), ('Q', Q), ('U', U), ('E', E), ('B', B)]:
        hp.write_map(
            os.path.join(out_dir, f'{comp}_{tag}.fits'),
            m, dtype=np.float32, overwrite=True,
        )

print('Helper functions defined.')

## 3. Reference Spectra

Compute the Planck 2018 best-fit spectra as a visual reference. These are the same $C_\ell$ arrays used in `generate_cmb_maps_theory_baseline`.

In [ ]:
sp_ref = generate_unlensed_spectra(
    planck_bf['H0'], planck_bf['ombh2'], planck_bf['omch2'],
    planck_bf['mnu'], planck_bf['omk'], planck_bf['tau'],
    planck_bf['As'], planck_bf['ns'], lmax,
)

print(f'Reference spectra: ell = [{sp_ref.ell[0]}, {sp_ref.ell[-1]}]')
print(f'TT peak Dl = {sp_ref.tt.max():.1f} μK²  at ell = {sp_ref.ell[sp_ref.tt.argmax()]}')

fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)
axes[0].plot(sp_ref.ell[2:], sp_ref.tt[2:], 'C0', lw=1.5)
axes[0].set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
axes[0].set_yscale('log')
axes[0].set_title('Planck 2018 best-fit — unlensed theoretical power spectra (reference)')
axes[1].plot(sp_ref.ell[2:], sp_ref.te[2:], 'C1', lw=1.5)
axes[1].axhline(0, color='k', lw=0.5, ls='--')
axes[1].set_ylabel(r'$D_\ell^{TE}\ [\mu K^2]$')
axes[2].plot(sp_ref.ell[2:], sp_ref.ee[2:], 'C2', lw=1.5)
axes[2].set_ylabel(r'$D_\ell^{EE}\ [\mu K^2]$')
axes[2].set_xlabel(r'Multipole $\ell$')
axes[2].set_yscale('log')
for ax in axes:
    ax.set_xlim(2, lmax)
plt.tight_layout()
plt.show()

## 4. Set A — Planck 2018 Posterior → `theory_planck_posterior`

1000 realisations drawn directly from the **Planck 2018 MCMC chains**
(`base_plikHM_TTTEEE_lowl_lowE`). This samples the full 6-parameter joint posterior — all
inter-parameter correlations (e.g. the $A_s$–$\tau$ degeneracy, the $H_0$–$\Omega_c h^2$
anti-correlation) are automatically respected.

The chains contain ~24 500 weighted samples after 30 % burn-in removal. We draw
$N_{\rm real} = 1000$ rows without replacement, using the chain multiplicities as sampling
weights. The six parameters extracted are:

| Parameter | Chain name | Note |
|---|---|---|
| H₀ | `H0` | derived parameter in the chains |
| Ω_b h² | `omegabh2` | sampled parameter |
| Ω_c h² | `omegach2` | sampled parameter |
| τ | `tau` | sampled parameter |
| ln(10¹⁰Aₛ) | `logA` | sampled parameter; converted via $A_s = e^{\rm logA} \times 10^{-10}$ |
| nₛ | `ns` | sampled parameter |

$m_\nu = 0.06\,{\rm eV}$ and $\Omega_k = 0$ are fixed (not varied in this chain).

In [ ]:
# ── Load Planck MCMC chains ───────────────────────────────────────────────────
_chain_root = os.path.join(
    '/nvme/h/lchristodoulou/data_p318',
    'COM_CosmoParams_base-plikHM-TTTEEE-lowl-lowE_R3',
    'base/plikHM_TTTEEE_lowl_lowE',
    'base_plikHM_TTTEEE_lowl_lowE',
)
_planck_chains = loadMCSamples(_chain_root, settings={'ignore_rows': 0.3})
_p = _planck_chains.getParams()
print(f'Chain samples after burn-in: {len(_planck_chains.samples):,}')

# ── Draw N_real rows without replacement, weighted by chain multiplicities ────
rng_a   = np.random.default_rng(seed + 1000)
_w      = _planck_chains.weights
_w      = _w / _w.sum()
_idx_a  = rng_a.choice(len(_planck_chains.samples), size=N_real, replace=False, p=_w)

H0_a    = _p.H0[_idx_a]
ombh2_a = _p.omegabh2[_idx_a]
omch2_a = _p.omegach2[_idx_a]
tau_a   = _p.tau[_idx_a]
lnAs_a  = _p.logA[_idx_a]
ns_a    = _p.ns[_idx_a]

print(f'\nSampled parameter ranges (N={N_real}):')
for name, arr in [('H0', H0_a), ('ombh2', ombh2_a), ('omch2', omch2_a),
                  ('tau', tau_a), ('logA', lnAs_a), ('ns', ns_a)]:
    print(f'  {name:8s}  mean={arr.mean():.5f}  std={arr.std():.5f}  '
          f'[{arr.min():.5f}, {arr.max():.5f}]')

# ── Simulation loop ───────────────────────────────────────────────────────────
tag_base_a = f'planck_post_nside{nside}'
metadata_a = []

for i in tqdm(range(N_real), desc='Set A — Planck posterior'):
    seed_i = seed + i
    As_i   = np.exp(lnAs_a[i]) * 1e-10
    sp = generate_unlensed_spectra(
        H0_a[i], ombh2_a[i], omch2_a[i],
        planck_bf['mnu'], planck_bf['omk'], tau_a[i],
        As_i, ns_a[i], lmax,
    )
    cl_tt_i = dl_to_cl(sp.ell, sp.tt)
    cl_ee_i = dl_to_cl(sp.ell, sp.ee)
    cl_te_i = dl_to_cl(sp.ell, sp.te)
    make_and_save_theory_maps(
        cl_tt_i, cl_ee_i, cl_te_i,
        nside, seed_i, out_planck_post, f'{tag_base_a}_{i:04d}',
    )
    metadata_a.append({
        'i': i, 'seed': seed_i,
        'H0': H0_a[i], 'ombh2': ombh2_a[i], 'omch2': omch2_a[i],
        'tau': tau_a[i], 'lnAs': lnAs_a[i], 'As': As_i, 'ns': ns_a[i],
    })

meta_path_a = os.path.join(out_planck_post, f'metadata_planck_post_nside{nside}.csv')
with open(meta_path_a, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metadata_a[0].keys()))
    writer.writeheader()
    writer.writerows(metadata_a)

print(f'\nDone. {N_real} × 5 maps → {out_planck_post}/')
print(f'Metadata  → {meta_path_a}')

## 5. Set B — DESI 2024 w0waCDM Posterior → `theory_desi_w0wa`

1000 realisations with CPL dark energy $w(a) = w_0 + w_a(1-a)$. All cosmological parameters
are drawn jointly from the **DESI 2024 MCMC chains**
(`base_w_wa-desi-bao-all_planck2018-lowl-TT-clik_planck2018-lowl-EE-clik_planck-NPIPE-highl-CamSpec-TTTEEE`).
This is a Cobaya run combining:

- DESI 2024 BAO (all tracers)
- Planck 2018 lowl TT + EE
- Planck NPIPE highl CamSpec TTTEEE

Sampling directly from the chains respects all inter-parameter correlations (e.g. the
$w_0$–$w_a$ anti-correlation, the $A_s$–$\tau$ degeneracy) without any Gaussian approximation.

The four Cobaya chain files (`chain.{1–4}.txt`) are read with pandas; 30 % burn-in is removed
before weighted sampling. The eight parameters extracted are:

| Parameter | Chain column | Note |
|---|---|---|
| w₀ | `w` | CPL $w_0$ |
| wₐ | `wa` | CPL $w_a$ |
| H₀ | `H0` | derived |
| Ω_b h² | `ombh2` | sampled |
| Ω_c h² | `omch2` | sampled |
| τ | `tau` | sampled |
| ln(10¹⁰Aₛ) | `logA` | sampled; converted via $A_s = e^{\rm logA} \times 10^{-10}$ |
| nₛ | `ns` | sampled |

$m_\nu = 0.06\,{\rm eV}$ and $\Omega_k = 0$ are fixed.

The parameter-sampling seed `seed + 2000` matches `generate_cmb_maps_planck` section 8.2.

In [ ]:
import pandas as pd

# ── Load DESI MCMC chains ─────────────────────────────────────────────────────
_desi_chain_dir = (
    '/nvme/h/lchristodoulou/data_p318/'
    'base_w_wa-desi-bao-all_planck2018-lowl-TT-clik_planck2018-lowl-EE-clik_planck-NPIPE-highl-CamSpec-TTTEEE'
)

def _read_cobaya_chain(path):
    with open(path) as f:
        cols = f.readline().lstrip('#').split()
    return pd.read_csv(path, sep=r'\s+', skiprows=1, names=cols)

_chain_files_b = [os.path.join(_desi_chain_dir, f'chain.{i}.txt') for i in range(1, 5)]
_desi_df = pd.concat([_read_cobaya_chain(p) for p in _chain_files_b], ignore_index=True)

# Apply 30% burn-in (consistent with Set A)
_burnin_b = int(0.3 * len(_desi_df))
_desi_df  = _desi_df.iloc[_burnin_b:].reset_index(drop=True)
print(f'DESI chain samples after burn-in: {len(_desi_df):,}')

# ── Draw N_real rows without replacement, weighted by multiplicities ───────────
rng_b   = np.random.default_rng(seed + 2000)
_w_b    = _desi_df['weight'].values
_w_b    = _w_b / _w_b.sum()
_idx_b  = rng_b.choice(len(_desi_df), size=N_real, replace=False, p=_w_b)

w0_samples = _desi_df['w'].values[_idx_b]
wa_samples = _desi_df['wa'].values[_idx_b]
H0_b       = _desi_df['H0'].values[_idx_b]
ombh2_b    = _desi_df['ombh2'].values[_idx_b]
omch2_b    = _desi_df['omch2'].values[_idx_b]
tau_b      = _desi_df['tau'].values[_idx_b]
lnAs_b     = _desi_df['logA'].values[_idx_b]
ns_b       = _desi_df['ns'].values[_idx_b]

print(f'\nSampled parameter ranges (N={N_real}):')
for name, arr in [('H0', H0_b), ('ombh2', ombh2_b), ('omch2', omch2_b),
                  ('tau', tau_b), ('logA', lnAs_b), ('ns', ns_b),
                  ('w0', w0_samples), ('wa', wa_samples)]:
    print(f'  {name:8s}  mean={arr.mean():.5f}  std={arr.std():.5f}  '
          f'[{arr.min():.5f}, {arr.max():.5f}]')

# Visualise w0-wa posterior samples (from chains)
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(w0_samples, wa_samples, s=2, alpha=0.4, c='C0')
ax.axvline(-1, color='k', lw=0.8, ls='--', label=r'$\Lambda$CDM ($w_0=-1, w_a=0$)')
ax.axhline(0,  color='k', lw=0.8, ls='--')
ax.set_xlabel(r'$w_0$')
ax.set_ylabel(r'$w_a$')
ax.legend(fontsize=8)
ax.set_title(f'DESI posterior samples (N={N_real})')
plt.tight_layout()
plt.show()

In [ ]:
tag_base_b = f'desi_w0wa_nside{nside}'
metadata_b = []

for i in tqdm(range(N_real), desc='Set B — DESI w0waCDM'):
    seed_i = seed + i
    As_i   = np.exp(lnAs_b[i]) * 1e-10
    sp = generate_unlensed_spectra(
        H0_b[i], ombh2_b[i], omch2_b[i],
        planck_bf['mnu'], planck_bf['omk'], tau_b[i],
        As_i, ns_b[i], lmax,
        w0=w0_samples[i], wa=wa_samples[i],
    )
    cl_tt_i = dl_to_cl(sp.ell, sp.tt)
    cl_ee_i = dl_to_cl(sp.ell, sp.ee)
    cl_te_i = dl_to_cl(sp.ell, sp.te)
    make_and_save_theory_maps(
        cl_tt_i, cl_ee_i, cl_te_i,
        nside, seed_i, out_desi_w0wa, f'{tag_base_b}_{i:04d}',
    )
    metadata_b.append({
        'i': i, 'seed': seed_i,
        'H0': H0_b[i], 'ombh2': ombh2_b[i],
        'omch2': omch2_b[i], 'tau': tau_b[i],
        'lnAs': lnAs_b[i], 'As': As_i, 'ns': ns_b[i],
        'w0': w0_samples[i], 'wa': wa_samples[i],
    })

meta_path_b = os.path.join(out_desi_w0wa, f'metadata_desi_w0wa_nside{nside}.csv')
with open(meta_path_b, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metadata_b[0].keys()))
    writer.writeheader()
    writer.writerows(metadata_b)

print(f'\nDone. {N_real} × 5 maps → {out_desi_w0wa}/')
print(f'Metadata  → {meta_path_b}')

## 6. Set C — Feature Model → `theory_feature`

1000 realisations with a sinusoidal feature superimposed on the primordial power spectrum:
$$P(k) = A_s\left(\frac{k}{0.05}\right)^{n_s-1}
\left[1 + A_{\rm lin}\,\sin(\phi + k\,f)\,e^{-(k-k_0)^2/\sigma^2}\right]$$
with $\phi=0$, $k_0=0.06$, $\sigma=0.04$ fixed. The ΛCDM base parameters are sampled from broad
uniform priors (same as `generate_cmb_maps_planck` section 8.3); $H_0$ and $\tau$ are fixed at
the Planck best-fit.

| Parameter | Range |
|---|---|
| Ω_c h² | [0.10, 0.15] |
| Ω_b h² | [0.021, 0.023] |
| Aₛ | [1.8, 2.4] × 10⁻⁹ |
| nₛ | [0.94, 0.99] |
| A_lin (feature amplitude) | [0.01, 0.06] |
| freq (feature frequency) | [20, 40] (integer) |

The parameter-sampling seed `seed + 3000` matches `generate_cmb_maps_planck` section 8.3.

In [ ]:
# Visualise feature vs ΛCDM at reference parameters before the full run
sp_feat_ex = generate_unlensed_spectra(
    planck_bf['H0'], planck_bf['ombh2'], planck_bf['omch2'],
    planck_bf['mnu'], planck_bf['omk'], planck_bf['tau'],
    planck_bf['As'], planck_bf['ns'], lmax,
    custom_PK=True, amp=0.03, freq=300, wid=0.04, centre=0.06, phase=0,
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sp_ref.ell[2:],      sp_ref.tt[2:],      'k-',  lw=2,   label='ΛCDM (Planck best-fit)')
ax.plot(sp_feat_ex.ell[2:],  sp_feat_ex.tt[2:],  'C3-', lw=1.5, alpha=0.8,
        label=r'Feature ($A_{\rm lin}=0.03$, $f=300$)')
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
ax.set_yscale('log')
ax.set_xlim(2, lmax)
ax.legend()
ax.set_title('TT: ΛCDM vs feature model example')
plt.tight_layout()
plt.show()

In [ ]:
rng_c = np.random.default_rng(seed + 3000)

_feat_lcdm = {
    'omch2': rng_c.uniform(0.10,   0.15,   N_real),
    'ombh2': rng_c.uniform(0.021,  0.023,  N_real),
    'As':    rng_c.uniform(1.8e-9, 2.4e-9, N_real),
    'ns':    rng_c.uniform(0.94,   0.99,   N_real),
}
A_lin_arr = rng_c.uniform(0.01, 0.06, N_real)
freq_arr   = rng_c.integers(20, 40,  N_real)

tag_base_c = f'feature_nside{nside}'
metadata_c = []

for i in tqdm(range(N_real), desc='Set C — Feature model'):
    seed_i = seed + i
    freq_i = int(freq_arr[i])
    sp = generate_unlensed_spectra(
        planck_bf['H0'], _feat_lcdm['ombh2'][i], _feat_lcdm['omch2'][i],
        planck_bf['mnu'], planck_bf['omk'], planck_bf['tau'],
        _feat_lcdm['As'][i], _feat_lcdm['ns'][i], lmax,
        custom_PK=True, amp=A_lin_arr[i], freq=freq_i, wid=0.04, centre=0.06, phase=0,
    )
    cl_tt_i = dl_to_cl(sp.ell, sp.tt)
    cl_ee_i = dl_to_cl(sp.ell, sp.ee)
    cl_te_i = dl_to_cl(sp.ell, sp.te)
    make_and_save_theory_maps(
        cl_tt_i, cl_ee_i, cl_te_i,
        nside, seed_i, out_feature, f'{tag_base_c}_{i:04d}',
    )
    metadata_c.append({
        'i': i, 'seed': seed_i,
        'H0': planck_bf['H0'], 'ombh2': _feat_lcdm['ombh2'][i],
        'omch2': _feat_lcdm['omch2'][i], 'tau': planck_bf['tau'],
        'As': _feat_lcdm['As'][i], 'ns': _feat_lcdm['ns'][i],
        'A_lin': A_lin_arr[i], 'freq': freq_i,
    })

meta_path_c = os.path.join(out_feature, f'metadata_feature_nside{nside}.csv')
with open(meta_path_c, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metadata_c[0].keys()))
    writer.writeheader()
    writer.writerows(metadata_c)

print(f'\nDone. {N_real} × 5 maps → {out_feature}/')
print(f'Metadata  → {meta_path_c}')

## 7. Sanity Checks

1. **File inventory** — correct number of FITS + CSV files per set
2. **Map properties** — nside, dtype, no NaN/Inf
3. **Statistics** — mean ≈ 0, std in expected range (T ~ 50–150 μK, Q/U/E ~ 1–5 μK, B ≈ 0)
4. **E/B ratio** — $\langle C_\ell^{EE}\rangle / \langle C_\ell^{BB}\rangle \gg 1$ (unlensed: B is pure numerical noise)
5. **Power spectrum** — pseudo-$C_\ell$ from T map consistent with theory
6. **Visual** — mollview of one realisation per set

In [ ]:
from pathlib import Path

_sets = {
    'planck_posterior': {
        'folder': out_planck_post,
        'tag':    f'planck_post_nside{nside}',
        'meta':   f'metadata_planck_post_nside{nside}.csv',
    },
    'desi_w0wa': {
        'folder': out_desi_w0wa,
        'tag':    f'desi_w0wa_nside{nside}',
        'meta':   f'metadata_desi_w0wa_nside{nside}.csv',
    },
    'feature': {
        'folder': out_feature,
        'tag':    f'feature_nside{nside}',
        'meta':   f'metadata_feature_nside{nside}.csv',
    },
}
_comps = ['T', 'Q', 'U', 'E', 'B']

# ── 1. File inventory ─────────────────────────────────────────────────────────
print('── 1. File inventory ──────────────────────────────────────────────────')
for name, info in _sets.items():
    n_fits = len(list(Path(info['folder']).glob('*.fits')))
    n_csv  = len(list(Path(info['folder']).glob('*.csv')))
    expected = N_real * len(_comps)
    ok = '✓' if n_fits == expected else f'✗ (expected {expected})'
    print(f"  {name:20s}  {n_fits:4d} FITS {ok}   {n_csv} CSV")

# ── 2 & 3. Map properties and statistics for realisation 0000 ─────────────────
print('\n── 2. Properties & 3. Statistics (realisation 0000) ─────────────────')
_maps0 = {}
for name, info in _sets.items():
    _maps0[name] = {}
    for comp in _comps:
        path = os.path.join(info['folder'], f"{comp}_{info['tag']}_0000.fits")
        _maps0[name][comp] = hp.read_map(path)
    T = _maps0[name]['T']
    ok_nside = hp.get_nside(T) == nside
    ok_float = T.dtype == np.float32
    ok_nan   = not np.isnan(T).any()
    print(f"\n  {name}")
    print(f"    nside={hp.get_nside(T)} {'✓' if ok_nside else '✗'}  "
          f"float32 {'✓' if ok_float else '✗'}  "
          f"NaN-free {'✓' if ok_nan else '✗'}")
    print(f"    {'comp':4s}  {'mean':>8s}  {'std':>8s}  μK")
    for comp in _comps:
        m = _maps0[name][comp]
        print(f"    {comp:4s}  {m.mean():+8.3f}  {m.std():8.3f}")

# ── 4. E/B power ratio ────────────────────────────────────────────────────────
print('\n── 4. E/B power ratio (ℓ = 50–500) ───────────────────────────────────')
for name in _sets:
    cl_E = hp.anafast(_maps0[name]['E'], lmax=500)[50:]
    cl_B = hp.anafast(_maps0[name]['B'], lmax=500)[50:]
    ratio = cl_E.mean() / (cl_B.mean() + 1e-30)
    flag = '✓' if ratio > 1e3 else '⚠ unexpectedly low'
    print(f"  {name:20s}  ⟨Cl_EE⟩ / ⟨Cl_BB⟩ = {ratio:.1e}  {flag}")

# ── 5. Power spectrum ─────────────────────────────────────────────────────────
print('\n── 5. Power spectrum vs Planck best-fit reference ─────────────────────')
_ell_v = np.arange(2, lmax + 1)
_dl_ref = sp_ref.tt[2:lmax + 1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(_ell_v, _dl_ref, 'k-', lw=2, label='Theory (Planck best-fit)', zorder=5)
for name, info in _sets.items():
    cl_pseudo = hp.anafast(_maps0[name]['T'], lmax=lmax)
    dl_pseudo  = _ell_v * (_ell_v + 1) / (2 * np.pi) * cl_pseudo[2:]
    ax.plot(_ell_v, dl_pseudo, alpha=0.7, label=name)
ax.set_xlabel(r'Multipole $\ell$')
ax.set_ylabel(r'$D_\ell^{TT}\ [\mu K^2]$')
ax.set_yscale('log')
ax.set_xlim(2, lmax)
ax.legend(fontsize=8)
ax.set_title('Pseudo-Cℓ (full sky, realisation 0000) vs Planck best-fit')
plt.tight_layout()
plt.show()

# ── 6. Metadata CSV ───────────────────────────────────────────────────────────
print('\n── 6. Metadata CSV ─────────────────────────────────────────────────────')
for name, info in _sets.items():
    df = pd.read_csv(os.path.join(info['folder'], info['meta']))
    seeds_ok = df['seed'].nunique() == len(df)
    print(f"  {name:20s}  {len(df)} rows  unique seeds {'✓' if seeds_ok else '✗'}")
    print(f"    columns: {list(df.columns)}")

In [ ]:
# ── 7. Visual: T map (realisation 0000) across all three sets ─────────────────
fig = plt.figure(figsize=(14, 9))
for row, (name, _) in enumerate(_sets.items()):
    for col, comp in enumerate(_comps):
        sub  = row * len(_comps) + col + 1
        m    = _maps0[name][comp]
        cmap = 'RdBu_r' if comp == 'T' else 'RdBu'
        hp.mollview(m, fig=fig, sub=(3, 5, sub),
                    title=f'{comp} — {name}', unit=r'$\mu K$',
                    cmap=cmap, margins=(0.005, 0.005, 0.005, 0.005))

plt.suptitle(
    'Realisation 0000 — T, Q, U, E, B across all sets\n'
    '(same seed → same random draw; differences reflect cosmology only)',
    fontsize=11, y=1.01,
)
plt.tight_layout()
plt.show()